# 02 - Prétraitement

Objectif : transformer les données brutes en jeux d'entraînement et de test propres.

**Ordre important :** nettoyage → split → normalisation (le scaler est ajusté sur le train seulement, sinon il y a *fuite de données*).

In [1]:
import sys; sys.path.append('..')
import pandas as pd
from src import preprocess as pp

## 1. Charger les 8 fichiers
Premier essai avec 50 000 lignes par fichier (rapide). Mets `None` à la fin pour tout charger.

In [2]:
df = pp.load_all(nrows_per_file=50_000)
print(df.shape)

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv          49,560 lignes
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv      48,766 lignes
Friday-WorkingHours-Morning.pcap_ISCX.csv                 48,847 lignes
Monday-WorkingHours.pcap_ISCX.csv                         49,107 lignes
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv   48,825 lignes
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv    49,032 lignes
Tuesday-WorkingHours.pcap_ISCX.csv                        47,867 lignes
Wednesday-workingHours.pcap_ISCX.csv                      48,955 lignes
(384956, 79)


## 2. Corriger les valeurs négatives impossibles
On repère les colonnes de débit/durée avec des valeurs négatives et on les remplace par 0.

In [3]:
df = pp.fix_negative_values(df)

  Flow Duration: 14 valeurs négatives -> 0
  Flow Bytes/s: 11 valeurs négatives -> 0
  Flow Packets/s: 14 valeurs négatives -> 0
  Flow IAT Mean: 14 valeurs négatives -> 0
  Flow IAT Max: 14 valeurs négatives -> 0
  Flow IAT Min: 700 valeurs négatives -> 0
  Fwd Header Length: 26 valeurs négatives -> 0
  Bwd Header Length: 19 valeurs négatives -> 0
  Fwd Header Length.1: 26 valeurs négatives -> 0


## 3. Créer l'étiquette binaire
`y_binary` : 0 = trafic normal, 1 = attaque. On garde aussi `Label` (multi-classe).

In [4]:
df = pp.add_binary_label(df)
print(df['y_binary'].value_counts())
print()
print(df['Label'].value_counts())

y_binary
0    353203
1     31753
Name: count, dtype: int64

Label
BENIGN                        353203
DDoS                           23864
FTP-Patator                     4082
DoS slowloris                   2532
Web Attack ï¿½ Brute Force      1060
PortScan                         213
Bot                                2
Name: count, dtype: int64


## 4. Split train/test + normalisation
Split **stratifié** 80/20 (mêmes proportions de classes), puis normalisation ajustée sur le train.

In [5]:
data = pp.split_and_scale(df)
print('train:', data['X_train'].shape, '| test:', data['X_test'].shape)
print('colonnes constantes supprimées:', data['dropped_constant'])
print('part d attaques train:', round(data['y_train'].mean(), 3), '| test:', round(data['y_test'].mean(), 3))

train: (307964, 68) | test: (76992, 68)
colonnes constantes supprimées: ['Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'CWE Flag Count', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']
part d attaques train: 0.082 | test: 0.082


## 5. Sauvegarder
Le fichier sera réutilisé par le notebook des modèles.

In [6]:
path = pp.save(data)
print('sauvegardé :', path)

sauvegardé : data\processed\dataset.joblib


## Observations
*(à compléter par toi)*

## Observations
- Après chargement de 8 fichiers (50 000 lignes chacun) et suppression des doublons : 384 956 flux, 79 colonnes.
- Le trafic est déséquilibré : environ 92 % de trafic normal (BENIGN) contre 8 % d'attaques. L'accuracy seule serait trompeuse, je regarderai le rappel et le F1-score.
- Des valeurs négatives impossibles (durées, débits, tailles d'en-têtes) ont été remplacées par 0. La colonne la plus touchée est Flow IAT Min (700 valeurs).
- Le split train/test est stratifié (80/20) : la part d'attaques est identique dans les deux jeux (8,2 %).
- La normalisation (StandardScaler) est ajustée sur le train uniquement, puis appliquée au test, pour éviter la fuite de données.
- 10 colonnes constantes (variance nulle) ont été supprimées, il reste 68 features.
- Limite : prendre les 50 000 premières lignes de chaque fichier donne un échantillon biaisé (par exemple seulement 2 flux « Bot » et aucune attaque Heartbleed ou SQL injection). Pour les résultats finaux, je chargerai les fichiers complets.